<a href="https://colab.research.google.com/github/computational-neurology/workshop2026public/blob/main/03_single_node_dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# @title Run this to install dependencies
%%capture
import sys
! pip install ipywidgets

In [11]:
# @title Imports, don't forget to run this cell
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp  # numerical integration solver

In [12]:
# @title Figure settings, don't forget to run this cell
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

import ipywidgets as widgets  # interactive display
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/nma.mplstyle")

In [13]:
# @title Plotting Functions

time = np.arange(0, 1, 0.01)

def plot_slope(dt):
  """
    Args:
      dt  : time-step
    Returns:
      A figure of an exponential, the slope of the exponential and the derivative exponential
  """

  t = np.arange(0, 5+0.1/2, 0.1)

  with plt.xkcd():

    fig = plt.figure(figsize=(6, 4))
    # Exponential
    p = np.exp(0.3*t)
    plt.plot(t, p, label='y')
    # slope
    plt.plot([1, 1+dt], [np.exp(0.3*1), np.exp(0.3*(1+dt))],':og',label=r'$\frac{y(1+\Delta t)-y(1)}{\Delta t}$')
    # derivative
    plt.plot([1, 1+dt], [np.exp(0.3*1), np.exp(0.3*(1))+dt*0.3*np.exp(0.3*(1))],'-k',label=r'$\frac{dy}{dt}$')
    plt.legend()
    plt.plot(1+dt, np.exp(0.3*(1+dt)), 'og')
    plt.ylabel('y')
    plt.xlabel('t')
    plt.show()



def plot_StepEuler(dt):
  """
    Args:
      dt  : time-step
    Returns:
      A figure of one step of the Euler method for an exponential growth function
  """

  t=np.arange(0, 1 + dt + 0.1 / 2, 0.1)

  with plt.xkcd():
    fig = plt.figure(figsize=(6,4))
    p=np.exp(0.3*t)
    plt.plot(t,p)
    plt.plot([1,],[np.exp(0.3*1)],'og',label='Known')
    plt.plot([1,1+dt],[np.exp(0.3*1),np.exp(0.3*(1))+dt*0.3*np.exp(0.3*1)],':g',label=r'Euler')
    plt.plot(1+dt,np.exp(0.3*(1))+dt*0.3*np.exp(0.3*1),'or',label=r'Estimate $p_1$')
    plt.plot(1+dt,p[-1],'bo',label=r'Exact $p(t_1)$')
    plt.vlines(1+dt,np.exp(0.3*(1))+dt*0.3*np.exp(0.3*1),p[-1],colors='r', linestyles='dashed',label=r'Error $e_1$')
    plt.text(1+dt+0.1,(np.exp(0.3*(1))+dt*0.3*np.exp(0.3*1)+p[-1])/2,r'$e_1$')
    plt.legend()
    plt.ylabel('Population (millions)')
    plt.xlabel('time(years)')
    plt.show()

def visualize_population_approx(t, p):
    fig = plt.figure(figsize=(6, 4))
    plt.plot(t, np.exp(0.3*t), 'k', label='Exact Solution')

    plt.plot(t, p,':o', label='Euler Estimate')
    plt.vlines(t, p, np.exp(0.3*t),
              colors='r', linestyles='dashed', label=r'Error $e_k$')

    plt.ylabel('Population (millions)')
    plt.legend()
    plt.xlabel('Time (years)')
    plt.show()

## LIF PLOT
def plot_IF(t, V, I, Spike_time):
  """
    Args:
      t  : time
      V  : membrane Voltage
      I  : Input
      Spike_time : Spike_times
    Returns:
      figure with three panels
      top panel: Input as a function of time
      middle panel: membrane potential as a function of time
      bottom panel: Raster plot
  """

  with plt.xkcd():
    fig = plt.figure(figsize=(12,4))
    gs = gridspec.GridSpec(3, 1,  height_ratios=[1, 4, 1])
    # PLOT OF INPUT
    plt.subplot(gs[0])
    plt.ylabel(r'$I_e(nA)$')
    plt.yticks(rotation=45)
    plt.plot(t,I,'g')
    #plt.ylim((2,4))
    plt.xlim((-50,1000))
    # PLOT OF ACTIVITY
    plt.subplot(gs[1])
    plt.plot(t,V,':')
    plt.xlim((-50,1000))
    plt.ylabel(r'$V(t)$(mV)')
    # PLOT OF SPIKES
    plt.subplot(gs[2])
    plt.ylabel(r'Spike')
    plt.yticks([])
    plt.scatter(Spike_time,1*np.ones(len(Spike_time)), color="grey", marker=".")
    plt.xlim((-50,1000))
    plt.xlabel('time(ms)')
    plt.show()

def plot_rErI(t, r_E, r_I):
  """
    Args:
      t   : time
      r_E : excitation rate
      r_I : inhibition rate

    Returns:
      figure of r_I and r_E as a function of time

  """
  with plt.xkcd():
    fig = plt.figure(figsize=(6,4))
    plt.plot(t,r_E,':',color='b',label=r'$r_E$')
    plt.plot(t,r_I,':',color='r',label=r'$r_I$')
    plt.xlabel('time(ms)')
    plt.legend()
    plt.ylabel('Firing Rate (Hz)')
    plt.show()


def plot_rErI_Simple(t, r_E, r_I):
  """
    Args:
      t   : time
      r_E : excitation rate
      r_I : inhibition rate

    Returns:
      figure with two panels
      left panel: r_I and r_E as a function of time
      right panel: r_I as a function of r_E with Nullclines

  """
  with plt.xkcd():
    fig = plt.figure(figsize=(12,4))
    gs = gridspec.GridSpec(1, 2)
    # LEFT PANEL
    plt.subplot(gs[0])
    plt.plot(t,r_E,':',color='b',label=r'$r_E$')
    plt.plot(t,r_I,':',color='r',label=r'$r_I$')
    plt.xlabel('time(ms)')
    plt.legend()
    plt.ylabel('Firing Rate (Hz)')
    # RIGHT PANEL
    plt.subplot(gs[1])
    plt.plot(r_E,r_I,'k:')
    plt.plot(r_E[0],r_I[0],'go')

    plt.hlines(0,np.min(r_E),np.max(r_E),linestyles="dashed",color='b',label=r'$\frac{d}{dt}r_E=0$')
    plt.vlines(0,np.min(r_I),np.max(r_I),linestyles="dashed",color='r',label=r'$\frac{d}{dt}r_I=0$')

    plt.legend(loc='upper left')

    plt.xlabel(r'$r_E$')
    plt.ylabel(r'$r_I$')
    plt.show()

def plot_rErI_Matrix(t, r_E, r_I, Null_rE, Null_rI):
  """
    Args:
      t   : time
      r_E : excitation firing rate
      r_I : inhibition firing rate
      Null_rE: Nullclines excitation firing rate
      Null_rI: Nullclines inhibition firing rate
    Returns:
      figure with two panels
      left panel: r_I and r_E as a function of time
      right panel: r_I as a function of r_E with Nullclines

  """

  with plt.xkcd():
    fig = plt.figure(figsize=(12,4))
    gs = gridspec.GridSpec(1, 2)
    plt.subplot(gs[0])
    plt.plot(t,r_E,':',color='b',label=r'$r_E$')
    plt.plot(t,r_I,':',color='r',label=r'$r_I$')
    plt.xlabel('time(ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.legend()
    plt.subplot(gs[1])
    plt.plot(r_E,r_I,'k:')
    plt.plot(r_E[0],r_I[0],'go')

    plt.plot(r_E,Null_rE,':',color='b',label=r'$\frac{d}{dt}r_E=0$')
    plt.plot(r_E,Null_rI,':',color='r',label=r'$\frac{d}{dt}r_I=0$')
    plt.legend(loc='best')
    plt.xlabel(r'$r_E$')
    plt.ylabel(r'$r_I$')
    plt.show()

# Node-level dynamics

Now that we have a general idea of what a dynamical system is and how numerical solutions work, we will start to build our own model of the brain. Today we simulate the activity of a single brain region using a Python package called [Neurolib](https://github.com/neurolib-dev/neurolib) allowing for easy simulations of brain functional activity.

Neurolib is very fast and easy to use, but there is also other software allowing you to do the same kinds of simulations, like [The Virtual Brain](https://thevirtualbrain.org/tvb/zwei) (TVB). With TVB you also get a graphical user interface if you are yet not too keen on programming, but want to explore whole-brain models while improving your coding skills. The only drawback from TVB is that it is currently pretty slow.

The same concepts you will learn today, though, also apply to TVB or any other whole-brain modeling packages.

# Kuramoto Model
*20 minutes*

In [14]:
%%capture
# @title run this cell so neurolib will be installed while you read the text
!{sys.executable} -m pip install git+https://github.com/neurolib-dev/neurolib.git

We will start to simulate a single brain region using a simple phase oscillator model known as the Kuramoto model, telling us how the phase of the oscillator evolves over time. We will consider:

$$ \frac{d \theta_i}{dt} = \omega_i + \zeta_i$$

Here $\theta_i$ is the phase of oscillator $i$, $\omega_i$ is the natural frequency of oscillator $i$, $\zeta_i$ is the noise term.

We will use Neurolib to perform our simulations.

In [15]:
import numpy as np
import matplotlib.pyplot as plt
# Some useful functions are provided here
import neurolib.utils.functions as func
# Let's import the Kuramoto model
from neurolib.models.kuramoto import KuramotoModel

# Initialize the Kuramoto model for a single node
model = KuramotoModel()
# Set the duration of your simulation
model.params['duration'] = 100
# Set the intrinsic frequency
model.params['omega'] = np.array([.5 * np.pi])
# Set the noise
model.params['sigma_ou'] = 0
# Let's run the model!
model.run()

### Perform exercise 1a-c
If you have time you can also move to the bonus exercise on coupled oscillators. Don't worry if you don't have time, it will also be part of the preparatory material for next week.

<div class="alert alert-block alert-success">
<b> Exercise 1a </b><p>

What are the model outputs? What do they mean? Hint: model.outputs is a dictionary...
<p>
</div>

In [16]:
print(...)

Ellipsis


<div class="alert alert-block alert-success">
<b> Exercise 1b </b><p>

Print the evolution of the oscillator's phase over time. Your code goes into [...]
<p>
</div>

In [ ]:
# Get the theta and cap it to only 2 pi ()
theta = model['theta'].T
theta_capped = np.mod(theta, 2*np.pi) # cap theta to [0, 2*pi]
# Plot the phases over time
plt.figure(figsize=(5,5))
plt.plot([...], [...])
plt.xlabel("Time")
plt.ylabel("Theta")
plt.yticks(np.arange(0, 2*np.pi+0.1, np.pi/2), [ r"$0$", r"$\pi/2$", r"$\pi$", r"$3/4\pi$", r"$2\pi$",])# modify y-axis ticks to be in multiples of pi
plt.show()

<div class="alert alert-block alert-success">
<b> Exercise 1c </b><p>

Run the following cell to display the outputs of the Kuramoto model for a single node.
Write down your answer to the following questions and then check for yourself:
1. What happens if you increase the frequency of the oscillator?
2. What happens if you decrease it?
3. What happens if you increase the noise?

Your answers:
1. ...
2. ...
3. ...

<p>
</div>

In [ ]:
# @title Run this cell to enable the widget
from ipywidgets import interact, FloatSlider
# Function to run the model and plot the results
def run_kuramoto_model(frequency=0.5*np.pi, noise=0):
    model = KuramotoModel()
    model.params['duration'] = 100
    model.params['omega'] = np.array([frequency])
    model.params['sigma_ou'] = noise
    model.run()
    plt.plot(model.outputs['t'], model.outputs['theta'].T)
    plt.title('Kuramoto Model Output')
    plt.xlabel('Time')
    plt.ylabel('Theta')
    plt.show()

# Create interactive widgets for frequency and noise
frequency_slider = FloatSlider(min=0, max=np.pi, step=0.1, value=0.5*np.pi, description='Frequency')
noise_slider = FloatSlider(min=0, max=2, step=0.1, value=0, description='Noise')

# Use the interact function to update the model based on slider values
interact(run_kuramoto_model, frequency=frequency_slider, noise=noise_slider)

## Bonus: Coupling oscillators together

The Kuramoto model for coupled oscillators is defined by the following differential equation:
$$ \frac{d \theta_i}{dt} = \omega_i + \zeta_i + \frac{K}{N} \sum_{j=1}^N A_{ij} sin(\theta_j(t - \tau_{ij}) - \theta_i(t)) + h_i(t)$$
here, as before, $\theta_i$ is the phase of oscillator $i$, $\omega_i$ is the natural frequency of oscillator $i$, $\zeta_i$ is the noise term, $K$ is the global coupling strength, $A$ is the coupling matrix, $\tau_{ij}$ is the phase lag between oscillator $i$ and $j$, and $h_i(t)$ is the external input to oscillator $i$.

 Coupling oscillators. Nodes in the network are influenced not only by their own natural frequency but also by the other nodes in the network. The strength of this influence is determined by the global coupling and the connectivity matrix. The degree of synchronization depends on the strength of the coupling. The Kuramoto model is relatively simple, mathematically tractable, and easy to understand.

Here we are dealing with phase over time, so this is not the same as the fMRI timeseries. Nonetheless, the same visual results would still apply to the amplitude of the signal. For example, if two region have the same, or very similar, phases, their functional connectivity would be ~1. Remember that functional connectivity measures how two regions behave together, meaning that if their amplitutes/phases increase at the same time, the regions have high functional connectivity, while as if one increases and the other one decreases, there is little functional connectivity between the two.


Turn off your volume before running the next video

In [20]:
# @title Run to show video of coupled oscillators
from IPython.display import IFrame

IFrame(src="https://www.youtube.com/watch?v=T58lGKREubo", width=560, height=315)

In the video you can see how oscillators behave when they are connected. What do you notice? How does the movement of each pendulum start with respect to others? How does it end?

Now we are going to explore the role of coupling. In this scenario we have several oscillators connected to each other. Each oscillator starts with its own frequency and then we will see how they behave over time. First, let's see what happens when the oscillators are not coupled. What do you see?

In [ ]:
# setting parameters
N = 6
# cmat is the structural connectivity matrix (here we start with a fully
# connected network)
cmat = np.ones((N, N)) # fully connected network
# dmat is the delay matrix, at the moment we consider instantaneous interactions
# between brain regions, so no delays
dmat = np.zeros((N,N)) # no delays

# Let's create a network model!
network_model = KuramotoModel(Cmat=cmat, Dmat=dmat)
# Set the duration of the simulation
network_model.params['duration'] = 50
# Set the intrinsic frequency
network_model.params['omega'] = np.random.rand((N)) * 0.25 * np.pi
# Let's start without any noise
network_model.params['sigma_ou'] = 0.
# For now our oscillators are uncoupled, so K = 0
network_model.params['k'] = 0
network_model.run()

theta = network_model['theta'].T
# cap the phase to be between 0 and 2pi
theta_capped = np.mod(theta, 2*np.pi)

# set up the figure
fig, ax = plt.subplots(1, 1, figsize=(10, 4))

plt.plot(network_model.t, theta_capped)
plt.xlabel("Time")
plt.ylabel("Theta")
plt.yticks(np.arange(0, 2*np.pi+0.1, np.pi/2), [ r"$0$", r"$\pi/2$", r"$\pi$", r"$3/4\pi$", r"$2\pi$",])# modify y-axis ticks to be in multiples of pi
plt.show()

<div class="alert alert-block alert-success">
<b> Exercise </b><p>

Change the coupling and noise to see what happens to the phases. What happens to the signals when you increase coupling? If you leave the coupling constant and increase noise, what happens? Does an increase in noise facilitate coupling or not?

Write down your answer...
<p>
</div>

In [22]:
# @title Run this cell to enable the widget
# Function to run the model and plot the results
def run_kuramoto_network_model(coupling, noise=0):
  # Let's create a network model!
  network_model = KuramotoModel(Cmat=cmat, Dmat=dmat)
  # Set the duration of the simulation
  network_model.params['duration'] = 50
  # Set the intrinsic frequencye
  network_model.params['omega'] = np.ones((N)) * 0.25 * np.pi
  # Let's start without any noise
  network_model.params['sigma_ou'] = noise
  # For now our oscillators are uncoupled, so K = 0
  network_model.params['k'] = coupling
  network_model.run()
  plt.plot(network_model.outputs['t'], network_model.outputs['theta'].T)
  plt.title('Kuramoto Model Output')
  plt.xlabel('Time')
  plt.ylabel('Theta')
  plt.yticks(np.arange(0, 2*np.pi+0.1, np.pi/2), [ r"$0$", r"$\pi/2$", r"$\pi$", r"$3/4\pi$", r"$2\pi$",])# modify y-axis ticks to be in multiples of pi
  plt.show()

# Create interactive widgets for frequency and noise
coupling_slider = FloatSlider(min=0, max=1, step=0.01, value=0, description='Coupling')
noise_slider = FloatSlider(min=0, max=2, step=0.1, value=0, description='Noise')

# Use the interact function to update the model based on slider values
interact(run_kuramoto_network_model, coupling=coupling_slider, noise=noise_slider)

<function __main__.run_kuramoto_network_model(coupling, noise=0)>

# Recap


A possible way to simulate the activity of the brain over time is to use **whole-brain models**.

Whole-brain models encompass a collection of equations that depict the dynamics and interplay among neural populations across various brain regions. In simpler terms, you divide the brain in different regions (remember the parcellations of different colors from the previous lecture) and simulate the activity of each region over time using a differential equation (for those that don't have a maths background, don't be scared, we won't dive into the details). These equations can be either formulated based on an understanding of the biophysical mechanisms that underlie diverse brain activities (biophysical models, e.g., based on specific assumptions like neuronal membrane conductance), or they can be chosen based on the dynamics they generate (phenomenological models, they describe what we see pretty well, even though they don't arise from biological variables). We will dive further into the differences later on.
The local dynamics between different regions are then combined with estimates of anatomical connectivity networks obtained from in vivo measurements (via the structural connectivity matrix we discussed in the previous lecture).

Most whole-brain models thus consist of three fundamental components:

- *Local dynamics:* The activity of each brain region is typically determined by the selected local dynamics along with interaction terms involving other regions (e.g non-linear oscillators, neural field theory, neural mass models, etc.). As previously stated, the spatial scale is a limiting factor. If you were to simulate each neuron, you would have approximately 86 billions equations + 100 trillions of connection (this is if you summarize each neuron with just one free variable)... This is not achievable in real life, so if we want to simulate the whole brain we need to compromise on spatial scale. What we can do is simplify the activity of hundreds of millions of neurons (those composing each of our ROIs, see next) with just one (or a few) simple equation. So each of our regions will be in the order of 5-10 cm^2.

- *Brain parcellation:* A brain parcellation determines the number of regions and the spatial resolution at which the brain dynamics occur. This parcellation may include cortical, sub-cortical, and cerebellar regions.

- *Anatomical connectivity matrix:* This matrix defines the network of connections between brain regions. Many studies rely on the human connectome, which is derived by estimating the number of white-matter fibers connecting different brain areas using DTI data combined with probabilistic tractography.


In this tutorial we covered some of the key components involved in computational modelling of single region dynamics. In the next lecture we will try to recreate a whole brain!

# Acknowledgments
Parts of this tutorial were taken from [Neuromatch Academy](https://compneuro.neuromatch.io/tutorials/intro.html). Their course on Computational Neuroscience delves into many more details on dynamical systems and biophysically detailed neural mass models. If you are interested in these topics, it is free to follow at your own pace.

[TVB](https://wiki.ebrains.eu/bin/view/Collabs/documentation/tutorials/The%20Virtual%20Brain/) also offers some more tutorials. Further tutorials  (credits to Davide Momi, Joana Cabral and John Griffiths) can be found [here](https://griffithslab.github.io/OHBM-whole-brain-modelling-course/).

